# Building Graph from OBJ
Simple graph representation of the Rhino building model.
References: S02-06, S02-03, S04-09

## 1. Import libraries

In [ ]:
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Graph import Graph

## 2. Check version

In [ ]:
print(Helper.Version())

## 3. Set renderer

In [ ]:
renderer = "vscode"

## 4. Import OBJ file
*(Ref: S02-06 § 4)*

In [ ]:
objects = Topology.ByOBJPath(r"C:\Users\etmaglari\IAAC\etmaglari_gML\Rhino_Geometry _etm.obj")
print(f"Imported: {objects}")

## 5. Show the geometry
*(Ref: S02-06 § 6)*

In [ ]:
Topology.Show(objects,
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 6. Extract top-level objects and compute bounding boxes
*(Ref: S04-09 § 7)*

The OBJ is surface geometry (no closed volumes). We use `Cluster.Contents` to get the meaningful top-level objects, then wrap each in a **bounding box** as a spatial proxy.

| Graph element | Represents |
|---|---|
| **Node** | One object / space (bounding-box centroid) |
| **Edge** | Spatial relationship: `touches` or `overlaps` |

In [ ]:
cluster = objects[0]

# Get top-level contents of the cluster (not individual faces)
contents = Cluster.Contents(cluster)
print(f"Top-level objects: {len(contents)}")

# Wrap each object in a bounding box as a spatial proxy
boxes = [Topology.BoundingBox(t) for t in contents]
print(f"Bounding boxes ready: {len(boxes)}")

## 7. Generate the spatial relationships graph
*(Ref: S04-09 § 9 — `Graph.BySpatialRelationships`)*

In [ ]:
graph = Graph.BySpatialRelationships(boxes, include=["touches", "overlaps"])
print(f"Graph: {len(Graph.Vertices(graph))} nodes, {len(Graph.Edges(graph))} edges")

## 8. Style the graph
*(Ref: S02-03 § 7)*

In [ ]:
for v in Graph.Vertices(graph):
    d = Dictionary.ByKeysValues(["size", "color"], [16, "red"])
    v = Topology.SetDictionary(v, d)

for e in Graph.Edges(graph):
    d = Dictionary.ByKeysValues(["width", "color"], [4, "white"])
    e = Topology.SetDictionary(e, d)

## 9. Visualise geometry + graph
*(Ref: S02-03 § 8)*

Red dots = spaces (nodes). White lines = adjacency between spaces (edges).

In [ ]:
Topology.Show(boxes, graph,
              faceColor=[210, 210, 250],
              faceOpacity=0.2,
              sagitta=0.15,
              absolute=False,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              backgroundColor="black",
              width=900, height=700,
              renderer=renderer)